In [1]:
import os
import sys
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain
from dotenv import load_dotenv
from tqdm import tqdm
import ast
import pandas as pd
import time
import random

In [2]:
curr_dir = os.getcwd()
parent_dir = os.path.dirname(curr_dir)
sys.path.append(parent_dir)

In [3]:
from prompt_templates.prompt_template import OpenEndedPromptTemplate
from database import MongoDBHelper
from utility.humaneval_functions import HumanEvalHelper
from llm_models.code_llms import Mistral, MetaLlama
from llm_models.code_reasoning_llms import DeepSeekLLM
from utility.mutation_functions import CodeMutator

## Loading various LLMs

In [4]:
load_dotenv()

zero_shot_template = OpenEndedPromptTemplate.zero_shot_prompt()

qn = '''def make_palindrome(string: str) -> str:

    def is_palindrome(string: str) -> bool:
        """ Test if given string is a palindrome """
        return string == string[::-1]'''
qn_desc = '''Find the shortest palindrome that begins with a supplied string.
Algorithm idea is simple:
- Find the longest postfix of supplied string that is a palindrome.
- Append to the end of the string reverse of a string prefix that comes before the palindromic suffix.'''

input_var = {
    "code": qn,
    "task": qn_desc
}


### Loading Mistral LLMs

In [5]:
mistral_llm = Mistral()
mistral_ans = mistral_llm.invoke(input_variables=input_var, prompt_template=zero_shot_template)
print(mistral_llm.process_ans(mistral_ans))

def make_palindrome(string: str) -> str:
    def is_palindrome(string: str) -> bool:
        """ Test if given string is a palindrome """
        return string == string[::-1]

    if not string:
        return ""

    n = len(string)
    for i in range(n):
        substring = string[i:]
        if is_palindrome(substring):
            return string + string[:i][::-1]

    return string + string[:-1][::-1]


### Loading and testing Llama 3.2 LLM

In [6]:
# llama_llm = MetaLlama()
# llama_ans = llama_llm.invoke(input_variables=input_var, prompt_template=zero_shot_template)
# print(llama_llm.process_ans(llama_ans))

### Loading and testing DeepSeekR1

In [7]:
# deepseek_llm = DeepSeekLLM("DeepSeekR1")
# deepseek_ans = deepseek_llm.invoke(input_variables=input_var, prompt_template=zero_shot_template)
# print(deepseek_llm.process_ans(deepseek_ans))

## Connecting to MongoDB

In [ ]:
db = MongoDBHelper()
db.check_database_connectivity()

ConfigurationError: The resolution lifetime expired after 21.133 seconds: Server Do53:192.168.1.1@53 answered The DNS operation timed out.; Server Do53:192.168.1.1@53 answered The DNS operation timed out.; Server Do53:192.168.1.1@53 answered The DNS operation timed out.; Server Do53:192.168.1.1@53 answered The DNS operation timed out.; Server Do53:192.168.1.1@53 answered The DNS operation timed out.; Server Do53:192.168.1.1@53 answered The DNS operation timed out.; Server Do53:192.168.1.1@53 answered The DNS operation timed out.

In [10]:
base_qns_db = db.client["Base_Questions_DB"]
open_ended_qns_db = base_qns_db["HumanEval_Open_Ended"]

In [ ]:
from prompt_templates.prompt_template import OpenEndedPromptTemplate
from utility.humaneval_functions import HumanEvalHelper
from utility.mutation_functions import CodeMutator

def run_test(prompt_technique: str, rounds: int = 50):

    ran_mutation_pass_count = 0
    doc_len = open_ended_qns_db.count_documents({})
    ran_failed_qn = {}
    all_qn = {}
    failed_validity = []
    llm = mistral_llm

    for i in tqdm(range(rounds)):
        task_id = f"HumanEvalo{43}"
        qn_sample = open_ended_qns_db.find_one({"_id": task_id})
        prompt_template = OpenEndedPromptTemplate.one_shot_prompt() if prompt_technique == "one shot" else OpenEndedPromptTemplate.few_shot_prompt()

        if qn_sample is not None:
            qn = qn_sample['qn']
            qn_desc = qn_sample['qn_desc']
            examples = qn_sample['examples']
            test_function = qn_sample['check']
            canon_sol = qn_sample['canon_solution']

            func_names, var_names = CodeMutator.obtain_key_info_from_code(qn)
            ran_mutation_source, ran_mutation_test = CodeMutator.mutate_task(source = qn, 
                                                examples = examples, 
                                                func_names=func_names, 
                                                check_function = test_function, 
                                                var_names=var_names,
                                                )

            prompt_examples = OpenEndedPromptTemplate.structure_one_shot_example(ran_mutation_test) if prompt_technique == "one shot" else OpenEndedPromptTemplate.structure_few_shot_examples(ran_mutation_test)

            input_var = {
                "code": ran_mutation_source,
                "task": qn_desc,
                "example": prompt_examples
            }

            complete_sol = qn + '\n' + canon_sol

            check_validity = HumanEvalHelper.check_test_case(test_case = test_function, code_snippet = complete_sol)        #Checks that the complete solution actually passes the test cases
            
            if check_validity is True:
                
                mutated_test_tree = ast.parse(random.choice(list(ran_mutation_test.keys())))

                for node in mutated_test_tree.body:
                    if isinstance(node, ast.Expr):
                        func_name = node.value.func.id
                
                ans = llm.invoke(input_variables= input_var, prompt_template=prompt_template)
                
                try: 
                    processed_output = llm.process_ans(ans)
                except ValueError:
                    ran_failed_qn[task_id] = ans

                all_qn[task_id] = processed_output

                try:
                    namespace = {}
                    exec(processed_output, namespace)
                    exec(test_function, namespace)
                    namespace['check'](namespace[func_name])
                    ran_mutation_pass_count += 1
                    print(f"{task_id}: {ran_mutation_pass_count}")
                except Exception as e:
                    print("=====")
                    print(f"----- LLM Output -----")
                    print(processed_output)
                    print("----- Complete Prompt -----")
                    print(prompt_template.format(**input_var))
                    print("----- Original Canon Solution -----")
                    print(complete_sol)
                    print("=====")
                    if isinstance(e, AssertionError):
                        print("{task_id}: Function failed to run due to following error -> {e}".format(e = e, task_id = task_id))
                        ran_failed_qn[task_id] = (processed_output, test_function)
                    else:
                        print("{task_id}: Could not run the LLM answer due to the following error {e}".format(e = e, task_id = task_id))
                        ran_failed_qn[task_id] = (processed_output, test_function)
            else: 
                print("{task_id}: Complete Solution failed the test function. Double check this entry.")
                failed_validity.append(task_id)

        else:
            pass
        if llm != mistral_llm:
            time.sleep(5)

In [31]:
run_test(prompt_technique= "few shot", rounds = 10)

 10%|█         | 1/10 [00:07<01:08,  7.60s/it]

HumanEvalo43: 1


 20%|██        | 2/10 [00:17<01:12,  9.08s/it]

HumanEvalo43: 2


 30%|███       | 3/10 [00:22<00:48,  6.94s/it]

...

HumanEvalo43: 6


 90%|█████████ | 9/10 [00:57<00:06,  6.45s/it]

HumanEvalo43: 7


100%|██████████| 10/10 [01:03<00:00,  6.38s/it]

HumanEvalo43: 8


## Zero-shot prompting

In [18]:
pass_count = 0
doc_len = open_ended_qns_db.count_documents({})
failed_qn = {}
all_qn = {}
failed_validity = []
llm = mistral_llm

for i in tqdm(range(50)):
    task_id = f"HumanEvalo{i}"
    qn_sample = open_ended_qns_db.find_one({"_id": task_id})

    if qn_sample is not None:
        qn = qn_sample['qn']
        qn_desc = qn_sample['qn_desc']
        examples = qn_sample['examples']
        test_function = qn_sample['check']
        canon_sol = qn_sample['canon_solution']

        input_var = {
            "code": qn,
            "task": qn_desc
        }
        
        complete_sol = qn + '\n' + canon_sol

        check_validity = HumanEvalHelper.check_test_case(test_case = test_function, code_snippet = complete_sol)        #Checks that the complete solution actually passes the test cases

        if check_validity is True:
            
            tree = ast.parse(complete_sol)

            for node in tree.body:
                if isinstance(node, ast.FunctionDef):
                    func_name = node.name
            
            complete_qn = qn + '\n' + qn_desc

            zero_shot_template = OpenEndedPromptTemplate.zero_shot_prompt()

            ans = llm.invoke(input_variables= input_var, prompt_template=zero_shot_template)
            
            try: 
                processed_output = llm.process_ans(ans)
            except ValueError:
                failed_qn[task_id] = ans

            all_qn[task_id] = processed_output

            try:
                namespace = {}
                exec(processed_output, namespace)
                exec(test_function, namespace)
                namespace['check'](namespace[func_name])
                pass_count += 1
            except Exception as e:
                print("=====")
                print(f"----- LLM Output -----")
                print(processed_output)
                print("----- Complete Prompt -----")
                print(zero_shot_template.format(**input_var))
                print("----- Original Canon Solution -----")
                print(complete_sol)
                print("=====")
                if isinstance(e, AssertionError):
                    print("{task_id}: Function failed to run due to following error -> {e}".format(e = e, task_id = task_id))
                    failed_qn[task_id] = (processed_output, test_function)
                else:
                    print("{task_id}: Could not run the LLM answer due to the following error {e}".format(e = e, task_id = task_id))
                    failed_qn[task_id] = (processed_output, test_function)
        else: 
            print("{task_id}: Complete Solution failed the test function. Double check this entry.")
            failed_validity.append(task_id)

    else:
        pass
    # time.sleep(5)

 32%|███▏      | 16/50 [00:23<01:04,  1.91s/it]

=====
----- LLM Output -----
from typing import List

def all_prefixes(string: str) -> List[str]:
    return [string[:i] for i in range(1, len(string) + 1)]
----- Complete Prompt -----

# Complete the given code snippet using the description below. Only complete the code function and do not add any other details.
Return a string containing space-delimited numbers starting from 0 upto n inclusive.

# Code Snippet:
def string_sequence(n: int) -> str:

# Your Answer: 

----- Original Canon Solution -----
def string_sequence(n: int) -> str:
    return ' '.join([str(x) for x in range(n + 1)])

=====
HumanEvalo15: Could not run the LLM answer due to the following error 'string_sequence'


 54%|█████▍    | 27/50 [00:45<00:40,  1.78s/it]

=====
----- LLM Output -----
from typing import List

def remove_duplicates(numbers: List[int]) -> List[int]:
    seen = set()
    result = []
    for num in numbers:
        if num not in seen:
            seen.add(num)
            result.append(num)
    return result
----- Complete Prompt -----

# Complete the given code snippet using the description below. Only complete the code function and do not add any other details.
From a list of integers, remove all elements that occur more than once.
Keep order of elements left the same as in the input.

# Code Snippet:
from typing import List

def remove_duplicates(numbers: List[int]) -> List[int]:

# Your Answer: 

----- Original Canon Solution -----
from typing import List

def remove_duplicates(numbers: List[int]) -> List[int]:
    import collections
    c = collections.Counter(numbers)
    return [n for n in numbers if c[n] <= 1]

=====
HumanEvalo26: Function failed to run due to following error -> 


 66%|██████▌   | 33/50 [01:03<00:48,  2.86s/it]

=====
----- LLM Output -----
def find_zero(xs: list):
    """
    Finds a zero of the polynomial with coefficients xs.
    Assumes xs has an even number of coefficients and largest non-zero coefficient is last.
    Returns only one zero point, even if there are multiple.
    """
    # Using the fact that for even degree polynomials with largest non-zero coefficient last,
    # there's a zero in the interval [-1, 1] due to Intermediate Value Theorem
    low, high = -1.0, 1.0
    while high - low > 1e-10:  # Using a small tolerance for convergence
        mid = (low + high) / 2
        if poly(xs, mid) * poly(xs, high) < 0:
            low = mid
        else:
            high = mid
    return (low + high) / 2
----- Complete Prompt -----

# Complete the given code snippet using the description below. Only complete the code function and do not add any other details.
xs are coefficients of a polynomial.
find_zero find x such that poly(x) = 0.
find_zero returns only only zero point, even if 

 92%|█████████▏| 46/50 [01:28<00:04,  1.19s/it]

=====
----- LLM Output -----
def change_base(x: int, base: int):
    if x == 0:
        return "0"
    digits = []
    while x > 0:
        digits.append(str(x % base))
        x = x // base
    return ''.join(reversed(digits))
----- Complete Prompt -----

# Complete the given code snippet using the description below. Only complete the code function and do not add any other details.
Given length of a side and high return area for a triangle.

# Code Snippet:
def triangle_area(a, h):

# Your Answer: 

----- Original Canon Solution -----
def triangle_area(a, h):
    return a * h / 2.0

=====
HumanEvalo45: Could not run the LLM answer due to the following error 'triangle_area'


100%|██████████| 50/50 [01:35<00:00,  1.92s/it]


In [19]:
print(f"Test cases passed for zero shot with Mistral Model: {pass_count}")

Test cases passed for zero shot with Mistral Model: 44


## Sequential Mutation Zero Shot Mistral

In [21]:
from prompt_templates.prompt_template import OpenEndedPromptTemplate
from utility.humaneval_functions import HumanEvalHelper
from utility.mutation_functions import CodeMutator

seq_mutation_pass_count = 0
doc_len = open_ended_qns_db.count_documents({})
seq_failed_qn = {}
all_qn = {}
failed_validity = []
llm = mistral_llm

for i in tqdm(range(50)):
    task_id = f"HumanEvalo{i}"
    qn_sample = open_ended_qns_db.find_one({"_id": task_id})

    if qn_sample is not None:
        qn = qn_sample['qn']
        qn_desc = qn_sample['qn_desc']
        examples = qn_sample['examples']
        test_function = qn_sample['check']
        canon_sol = qn_sample['canon_solution']

        func_names, var_names = CodeMutator.obtain_key_info_from_code(qn)
        seq_mutation_source, seq_mutation_test = CodeMutator.sequential_variable_mutation(source = qn, 
                                               examples = examples, 
                                               func_names=func_names, 
                                               check_function = test_function, 
                                               var_names=var_names
                                               )
        


        input_var = {
            "code": seq_mutation_source,
            "task": qn_desc
        }

        complete_sol = qn + '\n' + canon_sol

        check_validity = HumanEvalHelper.check_test_case(test_case = test_function, code_snippet = complete_sol)        #Checks that the complete solution actually passes the test cases
        
        if check_validity is True:
            
            if len(seq_mutation_test) > 0:
                mutated_test_tree = ast.parse(random.choice(list(seq_mutation_test.keys())))

                for node in mutated_test_tree.body:
                    if isinstance(node, ast.Expr):
                        func_name = node.value.func.id

            complete_qn = qn + '\n' + qn_desc

            zero_shot_template = OpenEndedPromptTemplate.zero_shot_prompt()

            ans = llm.invoke(input_variables= input_var, prompt_template=zero_shot_template)
            
            try: 
                processed_output = llm.process_ans(ans)
            except ValueError:
                seq_failed_qn[task_id] = ans

            all_qn[task_id] = processed_output

            try:
                namespace = {}
                exec(processed_output, namespace)
                exec(test_function, namespace)
                namespace['check'](namespace[func_name])
                seq_mutation_pass_count += 1
                print(f"{task_id}: {seq_mutation_pass_count}")
            except Exception as e:
                print("=====")
                print(f"----- LLM Output -----")
                print(processed_output)
                print("----- Complete Prompt -----")
                print(zero_shot_template.format(**input_var))
                print("----- Original Canon Solution -----")
                print(complete_sol)
                print("=====")
                if isinstance(e, AssertionError):
                    print("{task_id}: Function failed to run due to following error -> {e}".format(e = e, task_id = task_id))
                    seq_failed_qn[task_id] = (processed_output, test_function)
                else:
                    print("{task_id}: Could not run the LLM answer due to the following error {e}".format(e = e, task_id = task_id))
                    seq_failed_qn[task_id] = (processed_output, test_function)
        else: 
            print("{task_id}: Complete Solution failed the test function. Double check this entry.")
            failed_validity.append(task_id)

    else:
        pass
    if llm != mistral_llm:
        time.sleep(5)

  2%|▏         | 1/50 [00:01<01:02,  1.28s/it]

HumanEvalo0: 1


  4%|▍         | 2/50 [00:03<01:19,  1.66s/it]

HumanEvalo1: 2


  6%|▌         | 3/50 [00:04<01:18,  1.66s/it]

...

HumanEvalo47: 38


 98%|█████████▊| 49/50 [01:31<00:02,  2.80s/it]

=====
----- LLM Output -----
def generic_function1(var1: list):
    sorted_list = sorted(var1)
    n = len(sorted_list)
    if n == 0:
        return None
    if n % 2 == 1:
        return sorted_list[n // 2]
    else:
        return (sorted_list[n // 2 - 1] + sorted_list[n // 2]) / 2
----- Complete Prompt -----

# Complete the given code snippet using the description below. Only complete the code function and do not add any other details.
Checks if given string is a palindrome

# Code Snippet:
def generic_function1(var1: str):
    pass

# Your Answer: 

----- Original Canon Solution -----
def is_palindrome(text: str):
    for i in range(len(text)):
        if text[i] != text[len(text) - 1 - i]:
            return False
    return True

=====
HumanEvalo48: Function failed to run due to following error -> 


100%|██████████| 50/50 [01:33<00:00,  1.86s/it]

HumanEvalo49: 39


In [ ]:
print(seq_failed_qn)

{'HumanEvalo2': ('from typing import List\n\ndef generic_function1(var1: str) -> List[str]:\n    result = []\n    current_group = []\n    balance = 0\n\n    for char in var1.replace(" ", ""):\n        if char == \'(\':\n            if balance == 0:\n                current_group = []\n            balance += 1\n            current_group.append(char)\n        elif char == \')\':\n            balance -= 1\n            current_group.append(char)\n            if balance == 0:\n                result.append(\'\'.join(current_group))\n\n    return result', '\ndef check(candidate):\n    assert candidate(3.5) == 0.5\n    assert abs(candidate(1.33) - 0.33) < 1e-6\n    assert abs(candidate(123.456) - 0.456) < 1e-6'), 'HumanEvalo10': ('from typing import List, Tuple\n\ndef generic_function1(var1: List[int]) -> List[int]:\n    if not var1:\n        return []\n\n    rolling_max = []\n    current_max = var1[0]\n    rolling_max.append(current_max)\n\n    for num in var1[1:]:\n        if num > current_

In [22]:
print(f"Test cases passed for zero shot with Mistral Model: {seq_mutation_pass_count}")

Test cases passed for zero shot with Mistral Model: 39


## Random Mutation Zero Shot Mistral

In [26]:
from prompt_templates.prompt_template import OpenEndedPromptTemplate
from utility.humaneval_functions import HumanEvalHelper
from utility.mutation_functions import CodeMutator

ran_mutation_pass_count = 0
doc_len = open_ended_qns_db.count_documents({})
ran_failed_qn = {}
all_qn = {}
failed_validity = []
llm = mistral_llm

for i in tqdm(range(50)):
    task_id = f"HumanEvalo{i}"
    qn_sample = open_ended_qns_db.find_one({"_id": task_id})

    if qn_sample is not None:
        qn = qn_sample['qn']
        qn_desc = qn_sample['qn_desc']
        examples = qn_sample['examples']
        test_function = qn_sample['check']
        canon_sol = qn_sample['canon_solution']

        func_names, var_names = CodeMutator.obtain_key_info_from_code(qn)
        ran_mutation_source, ran_mutation_test = CodeMutator.random_variable_name_mutation(source = qn, 
                                               examples = examples, 
                                               func_names=func_names, 
                                               check_function = test_function, 
                                               var_names=var_names,
                                               generate_random_name=CodeMutator.generate_random_name
                                               )

        input_var = {
            "code": ran_mutation_source,
            "task": qn_desc
        }

        complete_sol = qn + '\n' + canon_sol

        check_validity = HumanEvalHelper.check_test_case(test_case = test_function, code_snippet = complete_sol)        #Checks that the complete solution actually passes the test cases
        
        if check_validity is True:
            
            mutated_test_tree = ast.parse(random.choice(list(ran_mutation_test.keys())))


            for node in mutated_test_tree.body:
                if isinstance(node, ast.Expr):
                    func_name = node.value.func.id
            
            complete_qn = qn + '\n' + qn_desc

            zero_shot_template = OpenEndedPromptTemplate.zero_shot_prompt()

            ans = llm.invoke(input_variables= input_var, prompt_template=zero_shot_template)
            
            try: 
                processed_output = llm.process_ans(ans)
            except ValueError:
                ran_failed_qn[task_id] = ans

            all_qn[task_id] = processed_output

            try:
                namespace = {}
                exec(processed_output, namespace)
                exec(test_function, namespace)
                namespace['check'](namespace[func_name])
                ran_mutation_pass_count += 1
                print(f"{task_id}: {ran_mutation_pass_count}")
            except Exception as e:
                print("=====")
                print(f"----- LLM Output -----")
                print(processed_output)
                print("----- Complete Prompt -----")
                print(zero_shot_template.format(**input_var))
                print("----- Original Canon Solution -----")
                print(complete_sol)
                print("=====")
                if isinstance(e, AssertionError):
                    print("{task_id}: Function failed to run due to following error -> {e}".format(e = e, task_id = task_id))
                    ran_failed_qn[task_id] = (processed_output, test_function)
                else:
                    print("{task_id}: Could not run the LLM answer due to the following error {e}".format(e = e, task_id = task_id))
                    ran_failed_qn[task_id] = (processed_output, test_function)
        else: 
            print("{task_id}: Complete Solution failed the test function. Double check this entry.")
            failed_validity.append(task_id)

    else:
        pass
    if llm != mistral_llm:
        time.sleep(5)

  2%|▏         | 1/50 [00:03<03:12,  3.92s/it]

HumanEvalo0: 1


  4%|▍         | 2/50 [00:06<02:37,  3.27s/it]

HumanEvalo1: 2


  6%|▌         | 3/50 [00:08<01:54,  2.43s/it]

...

HumanEvalo47: 40


 98%|█████████▊| 49/50 [02:07<00:02,  2.96s/it]

HumanEvalo48: 41


100%|██████████| 50/50 [02:09<00:00,  2.59s/it]

=====
----- LLM Output -----
def bEL(wfNwhkdBvG: int, nkwXbBGxJWB: int):
    if wfNwhkdBvG <= 0:
        return 0
    result = 1
    base = 2 % wfNwhkdBvG
    for _ in range(nkwXbBGxJWB):
        result = (result * base) % wfNwhkdBvG
    return result
----- Complete Prompt -----

# Complete the given code snippet using the description below. Only complete the code function and do not add any other details.
Return 2^n modulo p (be aware of numerics).

# Code Snippet:
def bEL(wfNwhkdBvG: int, nkwXbBGxJWB: int):
    pass

# Your Answer: 

----- Original Canon Solution -----
def modp(n: int, p: int):
    ret = 1
    for i in range(n):
        ret = (2 * ret) % p
    return ret

=====
HumanEvalo49: Function failed to run due to following error -> 


In [27]:
print(ran_mutation_pass_count)

41


In [ ]:
print(all_qn)

{'HumanEvalo0': 'from typing import List\n\ndef has_close_elements(numbers: List[float], threshold: float) -> bool:\n    numbers.sort()\n    for i in range(1, len(numbers)):\n        if abs(numbers[i] - numbers[i - 1]) < threshold:\n            return True\n    return False', 'HumanEvalo1': 'from typing import List\n\ndef separate_paren_groups(paren_string: str) -> List[str]:\n    paren_string = paren_string.replace(" ", "")\n    stack = []\n    result = []\n    current_group = []\n\n    for char in paren_string:\n        if char == \'(\':\n            if not stack:\n                current_group = []\n            stack.append(char)\n        elif char == \')\':\n            stack.pop()\n            if not stack:\n                result.append(\'\'.join(current_group))\n        current_group.append(char)\n\n    return result', 'HumanEvalo2': 'def truncate_number(number: float) -> float:\n    return number - int(number)', 'HumanEvalo3': 'from typing import List\n\ndef below_zero(operatio

In [ ]:
print(failed_qn)
pd.DataFrame(ran_failed_qn).to_csv(f"failed_qn_{'mistral' if llm == mistral_llm else 'llama'}.csv", index = False)

{'HumanEvalo1': ('from typing import List\n\ndef separate_paren_groups(paren_string: str) -> List[str]:\n    paren_string = paren_string.replace(" ", "")\n    stack = []\n    result = []\n    current_group = []\n\n    for char in paren_string:\n        if char == \'(\':\n            if not stack:\n                current_group = []\n            stack.append(char)\n        elif char == \')\':\n            stack.pop()\n            if not stack:\n                result.append(\'\'.join(current_group))\n        current_group.append(char)\n\n    return result', "\ndef check(candidate):\n    assert candidate('(()()) ((())) () ((())()())') == [\n        '(()())', '((()))', '()', '((())()())'\n    ]\n    assert candidate('() (()) ((())) (((())))') == [\n        '()', '(())', '((()))', '(((())))'\n    ]\n    assert candidate('(()(())((())))') == [\n        '(()(())((())))'\n    ]\n    assert candidate('( ) (( )) (( )( ))') == ['()', '(())', '(()())']"), 'HumanEvalo7': ('return [s for s in strin

# MCQ Zero Shot (Deprecated)

In [ ]:
template = OpenEndedPromptTemplate.zero_shot_prompt()

new_prompt = PromptTemplate(
    input_variables=["question", "test_case", "A", "B", "C", "D", "E"],
    template=template,
)

llm_chain = LLMChain(prompt=new_prompt, llm=deepseek_llm)

answer = llm_chain.run({
    "question":  question,
    "test_case": test_question,
    "A": options[0],
    "B": options[1],
    "C": options[2],
    "D": options[3],
    "E": options[4],
})


print(template.format(
    question = question,
    test_case = test_question,
    A = options[0],
    B = options[1],
    C = options[2],
    D = options[3],
    E = options[4]
))

NameError: name 'question' is not defined

In [ ]:
print(answer)

C) ayakkaya


In [ ]:
template = MCQPromptTemplate.zero_shot_prompt()

new_prompt = PromptTemplate(
    input_variables=["question", "test_case", "A", "B", "C", "D", "E"],
    template=template,
)

llm_chain = LLMChain(prompt=new_prompt, llm=deepseek_llm)

answers = {}

for idx in range(1, qn.count_documents({})+1):
    qn_id = f"HumanEval{idx}"
    print(qn_id)
    qn_sample = qn.find_one({"task_id": qn_id})
    question = qn_sample['prompt']
    test_question = qn_sample['question']
    raw_options = qn_sample["options"]
    if isinstance(raw_options, str):
        options = ast.literal_eval(raw_options)
    else:
        options = raw_options


    answer = llm_chain.run({
    "question":  question,
    "test_case": test_question,
    "A": options[0],
    "B": options[1],
    "C": options[2],
    "D": options[3],
    "E": options[4],
    })

    answers[qn_id] = answer

HumanEval1
HumanEval2
HumanEval3
HumanEval4
HumanEval5
HumanEval6
HumanEval7
HumanEval8
HumanEval9
HumanEval10
HumanEval11
HumanEval12
HumanEval13
HumanEval14
HumanEval15
HumanEval16
HumanEval17
HumanEval18
HumanEval19
HumanEval20


In [ ]:
for key in answers:
    print("ID: {id}, Answer: {answer}".format(id = key, answer = answers[key]))

ID: HumanEval1, Answer: D) 0.01
ID: HumanEval2, Answer: A) ['()', '(())', '(()())']
ID: HumanEval3, Answer: E) 0.556738
ID: HumanEval4, Answer: B) 3.666666667
ID: HumanEval5, Answer: B) [1, 2, 5, 2, 6, 8, 2, 10]
ID: HumanEval6, Answer: C) [3, 2, 1, 3]
ID: HumanEval7, Answer: A) ['acetoxyl', 'xyxyxy', 'epoxy', 'xylophone']
ID: HumanEval8, Answer: A) [40, -4200]
ID: HumanEval9, Answer: C) [10, 41, 41, 41, 41, 60]
ID: HumanEval10, Answer: E) kayayak
ID: HumanEval11, Answer: B) 11110
ID: HumanEval12, Answer: D) thisisnotthewaytopose
ID: HumanEval13, Answer: A) 5
ID: HumanEval14, Answer: D) ['b', 'be', 'bee', 'beep', 'beepe', 'beepee']
ID: HumanEval15, Answer: A) 0 1 2 3 4 5
ID: HumanEval16, Answer: C) 5
ID: HumanEval17, Answer: B) [4, 2, 1, 2, 2, 1, 1, 1, 4, 4, 4]
ID: HumanEval18, Answer: A) 3
ID: HumanEval19, Answer: A) 0 1 1 3 4 5 9
ID: HumanEval20, Answer: B) (2.0, 3.0)


# Mutation Testing
## Sequential Variable Naming Mutation

In [ ]:
from mutation_test import sequential_variable_mutation, random_variable_name_mutation, generate_random_name
from dataset_test import obtain_key_information

/Users/jin/Documents/GitHub/Code Reasoning Model Research Project/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Pinged your deployment. You successfully connected to MongoDB!
[]
Pinged your deployment. You successfully connected to MongoDB!
['separate_paren_groups'] ['paren_string'] ['current_string', 'result', 'c', 'current_depth']
from typing import List

def generic_function1(param1: str) -> List[str]:
    var2 = []
    var1 = []
    var4 = 0
    for var3 in param1:
        if var3 == '(':
            var4 += 1
            var1.append(var3)
        elif var3 == ')':
            var4 -= 1
            var1.append(var3)
            if var4 == 0:
                var2.append(''.join(var1))
                var1.clear()
    return var2
generic_function1(['( ) (( )) (( )( ))'])
from typing import List

def qVA(HpHuZh: str) -> List[str]:
    oqdFE = []
    uQCemsUDeOWz = []
    cHpuLOCaudSo = 0
    for CBScHAlx in HpHuZh:
        if CBScHAlx == '(':
            cHpuLOCaudSo += 1
            uQCemsUDeOWz.append(CBScHAlx)
        elif CBScHAlx == ')':
            cHpuLOCaudSo -= 1
            uQCemsUDeO

In [ ]:

seq_mutation_answers = {}

for idx in range(1, qn.count_documents({})+1):
    qn_id = f"HumanEval{idx}"
    print(qn_id)
    qn_sample = qn.find_one({"task_id": qn_id})
    question = qn_sample['prompt']
    test_question = qn_sample['question']
    raw_options = qn_sample["options"]
    if isinstance(raw_options, str):
        options = ast.literal_eval(raw_options)
    else:
        options = raw_options
    
    
    function_names, function_params, variable_names = obtain_key_information(question)

    mutated_question, mutated_test = sequential_variable_mutation(question, function_names, test_question, function_params, variable_names)

    print(mutated_question)
    print(mutated_test)

    
    answer = llm_chain.run({
    "question":  mutated_question,
    "test_case": mutated_test,
    "A": options[0],
    "B": options[1],
    "C": options[2],
    "D": options[3],
    "E": options[4],
    })

    seq_mutation_answers[qn_id] = answer

HumanEval1
from typing import List

def generic_function1(param2: List[float], var3: float) -> float:
    for var6, var5 in enumerate(param2):
        for var2, var4 in enumerate(param2):
            if var6 != var2:
                var1 = abs(var5 - var4)
                if var1 < var3:
                    var3 = var1
    return round(var3, 9)
generic_function1([[0.5, 0.2, 0.3, 0.55, 0.95, 0.1], 0.05])
HumanEval2
from typing import List

def generic_function1(param1: str) -> List[str]:
    var2 = []
    var1 = []
    var4 = 0
    for var3 in param1:
        if var3 == '(':
            var4 += 1
            var1.append(var3)
        elif var3 == ')':
            var4 -= 1
            var1.append(var3)
            if var4 == 0:
                var2.append(''.join(var1))
                var1.clear()
    return var2
generic_function1(['( ) (( )) (( )( ))'])
HumanEval3
def generic_function1(param1: float) -> float:
    return param1 % 1.0
generic_function1(3.556738)
HumanEval4
from typing 

In [ ]:
for key in seq_mutation_answers:
    print("ID: {id}, Answer: {answer}".format(id = key, answer = seq_mutation_answers[key]))

ID: HumanEval1, Answer: A) 0.05
ID: HumanEval2, Answer: A) ['()', '(())', '(()())']
ID: HumanEval3, Answer: E) 0.556738
ID: HumanEval4, Answer: B) 3.666666667
ID: HumanEval5, Answer: C) [1, 2, 2, 2, 5, 2, 6, 2, 8, 2, 10]
ID: HumanEval6, Answer: C) [3, 2, 1, 3]
ID: HumanEval7, Answer: A) ['acetoxyl', 'xyxyxy', 'epoxy', 'xylophone']
ID: HumanEval8, Answer: A) [40, -4200]
ID: HumanEval9, Answer: C) [10, 20, 30, 40, 41, 60]
ID: HumanEval10, Answer: B) kayaayak
ID: HumanEval11, Answer: B) 11110
ID: HumanEval12, Answer: A) pose
ID: HumanEval13, Answer: A) 5
ID: HumanEval14, Answer: D) ['b', 'be', 'bee', 'beep', 'beepe', 'beepee']
ID: HumanEval15, Answer: A) 0 1 2 3 4 5
ID: HumanEval16, Answer: E) 3
ID: HumanEval17, Answer: A) [4, 2, 1, 2, 2, 1, 2, 1, 1, 4, 4]
ID: HumanEval18, Answer: C) 4
ID: HumanEval19, Answer: B) zero one one three four five nine
ID: HumanEval20, Answer: B) (2.0, 3.0)


In [ ]:

ran_mutation_answers = {}

for idx in range(1, qn.count_documents({})+1):
    qn_id = f"HumanEval{idx}"
    print(qn_id)
    qn_sample = qn.find_one({"task_id": qn_id})
    question = qn_sample['prompt']
    test_question = qn_sample['question']
    raw_options = qn_sample["options"]
    if isinstance(raw_options, str):
        options = ast.literal_eval(raw_options)
    else:
        options = raw_options
    
    
    function_names, function_params, variable_names = obtain_key_information(question)

    mutated_question, mutated_test = random_variable_name_mutation(question, function_names, test_question, function_params, variable_names, generate_random_name)

    print(mutated_question)
    print(mutated_test)
    
    answer = llm_chain.run({
    "question":  mutated_question,
    "test_case": mutated_test,
    "A": options[0],
    "B": options[1],
    "C": options[2],
    "D": options[3],
    "E": options[4],
    })

    ran_mutation_answers[qn_id] = answer

HumanEval1
from typing import List

def MeRHJo(ySfMc: List[float], rMy: float) -> float:
    for XtUAdNEw, ABfqbkGtEsQqmD in enumerate(ySfMc):
        for SXzX, VHbAT in enumerate(ySfMc):
            if XtUAdNEw != SXzX:
                oYegCK = abs(ABfqbkGtEsQqmD - VHbAT)
                if oYegCK < rMy:
                    rMy = oYegCK
    return round(rMy, 9)
MeRHJo([[0.5, 0.2, 0.3, 0.55, 0.95, 0.1], 0.05])
HumanEval2
from typing import List

def xhvr(CuMKFa: str) -> List[str]:
    DYfQTVFWqdOUC = []
    zQQkWjdHCJzC = []
    pHaDUd = 0
    for sAa in CuMKFa:
        if sAa == '(':
            pHaDUd += 1
            zQQkWjdHCJzC.append(sAa)
        elif sAa == ')':
            pHaDUd -= 1
            zQQkWjdHCJzC.append(sAa)
            if pHaDUd == 0:
                DYfQTVFWqdOUC.append(''.join(zQQkWjdHCJzC))
                zQQkWjdHCJzC.clear()
    return DYfQTVFWqdOUC
xhvr(['( ) (( )) (( )( ))'])
HumanEval3
def otemxy(ixQLfJgtoQj: float) -> float:
    return ixQLfJgtoQj % 1.0
o

In [ ]:
for key in ran_mutation_answers:
    print("ID: {id}, Answer: {answer}".format(id = key, answer = ran_mutation_answers[key]))

ID: HumanEval1, Answer: C) 0.005
ID: HumanEval2, Answer: A) ['()', '(())', '(()())']
ID: HumanEval3, Answer: E) 0.556738
ID: HumanEval4, Answer: C) 5.333333333
ID: HumanEval5, Answer: B) [1, 2, 5, 2, 6, 8, 2, 10]
ID: HumanEval6, Answer: A) [2, 3, 1, 3]
ID: HumanEval7, Answer: A) ['acetoxyl', 'xyxyxy', 'epoxy', 'xylophone']
ID: HumanEval8, Answer: A) [40, -4200]
ID: HumanEval9, Answer: C) [10, 20, 30, 40, 41, 60]
ID: HumanEval10, Answer: E) kayayak
ID: HumanEval11, Answer: B) 11110
ID: HumanEval12, Answer: A) pose
ID: HumanEval13, Answer: A) 5
ID: HumanEval14, Answer: D) ['b', 'be', 'bee', 'beep', 'beepe', 'beepee']
ID: HumanEval15, Answer: A) 0 1 2 3 4 5
ID: HumanEval16, Answer: C) 5
ID: HumanEval17, Answer: A) [4, 2, 1, 2, 2, 1, 1, 1, 4, 4]
ID: HumanEval18, Answer: A) 2
ID: HumanEval19, Answer: B) zero one one three four five nine
ID: HumanEval20, Answer: B) (2.0, 3.0)
